In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pygwalker as pyg
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os
from sklearn.decomposition import PCA


In [17]:
#Importing feature_extraction.csv file

feat_df_v3 = pd.read_csv("../data/proccessed/feature_extraction_V3.csv")
feat_df_v4 = pd.read_csv("../data/proccessed/feature_extraction_V4.csv")
feat_df = feat_df_v4.merge(
    feat_df_v3[["file name", "label"]],
    on="file name",
    how="left"
)
print("Filas antes del merge:", len(feat_df_v4))
print("Filas después del merge:", len(feat_df))
print("Labels nulos tras el merge:", feat_df["label"].isnull().sum())

Filas antes del merge: 2503
Filas después del merge: 2503
Labels nulos tras el merge: 0


I have anormal values here, since i will be using random forest (explain details later); these will interfere with the model training (make better notes for the complete version of this notebook)

In [3]:
type(feat_df["label"][0])

numpy.int64

In [18]:
feat_df.shape

(2503, 20)

In [19]:
print(feat_df["repo"].nunique())
print(feat_df.groupby("repo").size().sort_values())

22
repo
flow                             2
Discord-OTP-Forcer              11
ibet-Network                    12
hass-dab-pumps                  16
cli                             25
kernelci-pipeline               29
ibet-SmartContract              31
index-digest                    35
FreeCAD-DFM-Workbench           54
mesa-llm                        61
SpiceyPy                        63
draftjs_exporter                68
pyticktick                      68
simonwillisonblog              131
moldflow-api                   151
XMRT-Ecosystem                 153
FAIR                           161
penny                          191
wicked-garden                  201
pypto                          289
commander-spellbook-backend    292
TileOPs                        459
dtype: int64


In [20]:
cols_relativas = ["commit_count", "author_count", "churn_rate", "days_since_change"]
print(feat_df[cols_relativas].describe())

       commit_count  author_count   churn_rate  days_since_change
count   2503.000000   2503.000000  2503.000000        2503.000000
mean       2.305496      1.234579     2.789575           1.304804
std        4.442194      0.748839    12.133475           1.688684
min        0.111111      0.250000     0.000000           0.060294
25%        0.500000      1.000000     0.442053           0.963636
50%        1.000000      1.000000     1.000000           1.000000
75%        2.000000      1.000000     2.237742           1.153846
max       77.000000      8.000000   458.924370          27.455285


In [21]:

#limit_ch = feat_df["churn_rate"].quantile(0.99)
#feat_df["churn_rate"] = feat_df["churn_rate"].clip(upper=limit_ch)

limit_mc = feat_df["max_complexity"].quantile(0.99)
feat_df["max_complexity"] = feat_df["max_complexity"].clip(upper=limit_mc)
print(feat_df["max_complexity"].max())

42.98000000000002


In [22]:
limit_churn_rel = feat_df["churn_rate"].quantile(0.99)
feat_df["churn_rate"] = feat_df["churn_rate"].clip(upper=limit_churn_rel)
print("Nuevo máximo de churn_rate:", feat_df["churn_rate"].max())

limit_commit_rel = feat_df["commit_count"].quantile(0.99)
feat_df["commit_count"] = feat_df["commit_count"].clip(upper=limit_commit_rel)
print("Nuevo máximo de commit_count:", feat_df["commit_count"].max())

Nuevo máximo de churn_rate: 27.046608695652186
Nuevo máximo de commit_count: 20.966666666666697


In [23]:
print(feat_df[["commit_count", "author_count", "churn_rate", "days_since_change"]].describe())

       commit_count  author_count   churn_rate  days_since_change
count   2503.000000   2503.000000  2503.000000        2503.000000
mean       2.167981      1.234579     2.307623           1.304804
std        3.257588      0.748839     4.005417           1.688684
min        0.111111      0.250000     0.000000           0.060294
25%        0.500000      1.000000     0.442053           0.963636
50%        1.000000      1.000000     1.000000           1.000000
75%        2.000000      1.000000     2.237742           1.153846
max       20.966667      8.000000    27.046609          27.455285


In [24]:
excluir = ["file name", "repo", "bug_fix_commits", "refactor_commits", "label"]
features_v2 = [c for c in feat_df.columns if c not in excluir]

In [25]:
X = feat_df[features_v2]
Y = feat_df["label"] 

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

In [27]:
modelo_v2_relativo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_v2_relativo.fit(X_train, y_train)
# Evaluación
y_pred = modelo_v2_relativo.predict(X_test)

print("\nScore:", modelo_v2_relativo.score(X_test, y_test))
print()
print(classification_report(y_test, y_pred))
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print()

importancias = pd.Series(modelo_v2_relativo.feature_importances_, index=features_v2)
print(importancias.sort_values(ascending=False))


Score: 0.874251497005988

              precision    recall  f1-score   support

           0       0.86      0.95      0.90       311
           1       0.89      0.76      0.82       190

    accuracy                           0.87       501
   macro avg       0.88      0.85      0.86       501
weighted avg       0.88      0.87      0.87       501

Matriz de confusión:
[[294  17]
 [ 46 144]]

commit_count             0.292385
author_count             0.086463
churn_rate               0.085921
days_since_change        0.069956
dependency_count         0.066958
loc                      0.051608
sloc                     0.050454
blank_ratio              0.042481
avg_function_len         0.039961
maintainability_index    0.035812
halstead_effort          0.034083
cyclomatic_complexity    0.031527
comment_ratio            0.031064
num_functions            0.030560
max_complexity           0.022298
max_nesting_depth        0.019806
is_tested                0.008663
dtype: float64


In [21]:
os.makedirs("../models", exist_ok=True)

joblib.dump(modelo_v2_relativo, "../models/random_forest_v2.pkl")
joblib.dump(features_v2, "../models/features_v2.pkl")

print("Guardado: random_forest_v2.pkl y features_v2.pkl")

Guardado: random_forest_v2.pkl y features_v2.pkl


In [29]:
embeddings = np.load("../data/proccessed/embeddings_por_archivo.npz")
print("Archivos en el npz:", len(embeddings.files))
print("Filas en feat_df (después del merge):", len(feat_df))

Archivos en el npz: 2503
Filas en feat_df (después del merge): 2503


In [30]:
claves_npz = set(embeddings.files)
claves_df = set(feat_df["file name"])

print("¿Son exactamente el mismo conjunto?", claves_npz == claves_df)
print("En df pero no en npz:", claves_df - claves_npz)
print("En npz pero no en df:", claves_npz - claves_df)

¿Son exactamente el mismo conjunto? False
En df pero no en npz: {'XMRT-Ecosystem/config\\__init__.py', 'penny/penny\\penny\\email\\protocol.py', 'pypto/tests\\ut\\ir\\memory\\test_memref.py', 'commander-spellbook-backend/backend\\spellbook\\admin\\card_admin.py', 'pypto/tests\\ut\\ir\\statements\\test_while_stmt.py', 'pyticktick/src\\pyticktick\\models\\v2\\parameters\\task_parent.py', 'moldflow-api/tests\\api\\unit_tests\\test_unit_string_array.py', 'wicked-garden/scripts\\_capability_registry.py', 'pypto/tests\\st\\runtime\\test_concat.py', 'FAIR/src\\fair_platform\\extensions\\core\\main.py', 'pypto/tests\\ut\\ir\\expressions\\test_comparison_ops.py', 'index-digest/indexdigest\\linters\\linter_0089_empty_tables.py', 'commander-spellbook-backend/backend\\spellbook\\tests\\test_views\\test_estimate_bracket.py', 'mesa-llm/examples\\negotiation\\model.py', 'moldflow-api/src\\moldflow\\constants.py', 'TileOPs/workloads\\ops\\gated_deltanet_recurrence.py', 'commander-spellbook-backend/bac

In [31]:
embeddings_dict = {k: embeddings[k] for k in embeddings.files}
feat_df["file name"] = feat_df["file name"].str.replace("\\", "/", regex=False)
matriz_embeddings = np.array([embeddings_dict[clave] for clave in feat_df["file name"]])

In [32]:
claves_npz = set(embeddings_dict.keys())
claves_df = set(feat_df["file name"])
print("¿Coinciden ahora?", claves_npz == claves_df)

¿Coinciden ahora? True


In [33]:
import numpy as np
from sklearn.decomposition import PCA

embeddings = np.load("../data/proccessed/embeddings_por_archivo.npz")

matriz_embeddings = np.array([embeddings[clave] for clave in feat_df["file name"]])

emb_train = matriz_embeddings[X_train.index]
emb_test = matriz_embeddings[X_test.index]

pca = PCA(n_components=10, random_state=42)
emb_train_reducido = pca.fit_transform(emb_train)
emb_test_reducido = pca.transform(emb_test)  

print("Varianza explicada por las 10 componentes:", pca.explained_variance_ratio_.sum())

Varianza explicada por las 10 componentes: 0.4119232047935397


In [34]:
pca_completo = PCA(n_components=50, random_state=42)
pca_completo.fit(emb_train)

varianza_acumulada = np.cumsum(pca_completo.explained_variance_ratio_)
for n in [5, 10, 15, 20, 30, 50]:
    print(f"{n} componentes -> {varianza_acumulada[n-1]:.2%} varianza acumulada")

5 componentes -> 31.18% varianza acumulada
10 componentes -> 41.19% varianza acumulada
15 componentes -> 48.70% varianza acumulada
20 componentes -> 54.20% varianza acumulada
30 componentes -> 62.27% varianza acumulada
50 componentes -> 72.51% varianza acumulada


In [35]:
pca_20 = PCA(n_components=20, random_state=42)
emb_train_20 = pca_20.fit_transform(emb_train)
emb_test_20 = pca_20.transform(emb_test)

# Combina con tus features tabulares existentes
X_train_final = np.hstack([X_train.values, emb_train_20])
X_test_final = np.hstack([X_test.values, emb_test_20])

modelo_v3 = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_v3.fit(X_train_final, y_train)

y_pred_v3 = modelo_v3.predict(X_test_final)
print("Score:", modelo_v3.score(X_test_final, y_test))
print(classification_report(y_test, y_pred_v3))

Score: 0.8682634730538922
              precision    recall  f1-score   support

           0       0.86      0.94      0.90       311
           1       0.88      0.75      0.81       190

    accuracy                           0.87       501
   macro avg       0.87      0.85      0.86       501
weighted avg       0.87      0.87      0.87       501



In [37]:
# El modelo entrenado con features tabulares + embeddings reducidos
joblib.dump(modelo_v3, "../models/random_forest_v3.pkl")

# El PCA ya ajustado — SIN esto, predict.py no puede transformar embeddings nuevos
# de forma consistente con lo que vio el modelo en entrenamiento
joblib.dump(pca_20, "../models/pca_v3.pkl")

# Lista de las columnas tabulares, en el orden exacto usado al construir X_train_final
joblib.dump(features_v2, "../models/features_v3.pkl")

print("Guardado:")
print("  models/random_forest_v3.pkl")
print("  models/pca_v3.pkl")
print("  models/features_v3.pkl")

Guardado:
  models/random_forest_v3.pkl
  models/pca_v3.pkl
  models/features_v3.pkl


In [38]:
modelo_cargado = joblib.load("../models/random_forest_v3.pkl")
pca_cargado = joblib.load("../models/pca_v3.pkl")

score_verificacion = modelo_cargado.score(X_test_final, y_test)
print("Score al recargar:", score_verificacion)

Score al recargar: 0.8682634730538922
